# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze a dataset defined by a [Croissant schema](https://mlcommons.org/croissant/) using the `mlcroissant` library, with all references made through entity `@id` fields as per best practices for FAIR data.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset via Croissant schema
dataset = mlc.Dataset(croissant_url)

# Access the metadata object (do not treat as dict)
meta = dataset.metadata

print(meta.name + ": " + meta.description)
print(f"Version: {meta.version}")
print(f"Identifier: {meta.identifier}")

## 2. Data Overview
Review the available record sets, fields, and their `@id`s.

We will print each record set's `@id`, name, and list its fields and columns (also by `@id`).

In [ ]:
# List all record sets and their details
print('Available record sets:')
for rs in meta.record_sets:
    print(f"- RecordSet @id: {rs.id} | name: {rs.name}")
    if hasattr(rs, 'description'):
        print(f"    description: {rs.description}")
    print("    Fields:")
    for field in getattr(rs, 'fields', []):
        print(f"        - Field @id: {field.id} | name: {getattr(field, 'name', '')} | dataType: {getattr(field, 'data_type', '')}")
    print("    Columns:")
    for col in getattr(rs, 'columns', []):
        print(f"        - Column @id: {col.id} | source: {getattr(col, 'source', None)}")
    print()

# For demonstration, show a few records from the first record set (by @id):
if meta.record_sets:
    first_rs_id = meta.record_sets[0].id
    print(f"First few records from record_set {first_rs_id}:")
    for x in dataset.records(record_set=first_rs_id):
        pprint.pprint(x)
        break  # Just print one for demonstration

## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis, using the record set `@id`s from the overview above.

> Note: All keys and columns are referenced by their `@id`.

In [ ]:
# Extract data from all record sets
record_sets_ids = [rs.id for rs in meta.record_sets]
dataframes = {}
for rs_id in record_sets_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records) if records else pd.DataFrame()

# Show columns for the first record set
if len(record_sets_ids) > 0:
    first_rs = record_sets_ids[0]
    print(f"Columns for record set {first_rs}:\n", dataframes[first_rs].columns.tolist())
    print(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing steps: filtering, normalization, categorization, and group analysis. All references are via entity `@id`.

Let's assume the main survey record set contains a field like `phq9_total_score` for PHQ-9 depression scores, and `village` for participant's village.

In [ ]:
# Inspect what columns are available in the primary record set
main_rs_id = record_sets_ids[0]  # Assume first listed record set is the primary survey data
df = dataframes[main_rs_id]
print(f"Available columns in {main_rs_id}:", df.columns.tolist())

# Depending on schema, adjust field IDs. We'll use common names for demonstration here:
numeric_field_id = next((col for col in df.columns if "phq9_total" in col.lower() or "phq9" in col.lower()), None)
if not numeric_field_id:
    numeric_field_id = df.columns[0]  # fallback
print(f"Using numeric field: {numeric_field_id}")

group_field_id = next((col for col in df.columns if "village" in col.lower() or "group" in col.lower()), None)
if not group_field_id:
    if len(df.columns) > 1:
        group_field_id = df.columns[1]
    else:
        group_field_id = numeric_field_id
print(f"Using group field: {group_field_id}")

# Filter for high PHQ-9 scores (e.g., >10)
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
else:
    # Try to convert to numeric if necessary
    filtered_df = df.copy()
    filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
    filtered_df = filtered_df[filtered_df[numeric_field_id] > 10]

print(f"Filtered records with {numeric_field_id} > 10:")
print(filtered_df.head())

# Normalize PHQ-9 scores
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group analysis: mean PHQ-9 by village
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_score')
    print(f"Mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

> Example: Distribution of PHQ-9 total scores and mean group differences.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for total scores
if numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    df[numeric_field_id].dropna().astype(float).hist(bins=20)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# Bar plot for group means
if group_field_id in df.columns and numeric_field_id in df.columns:
    group_means = df.groupby(group_field_id)[numeric_field_id].mean().dropna()
    plt.figure(figsize=(10,4))
    group_means.plot(kind='bar')
    plt.title(f'Mean {numeric_field_id} per {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we explored a FAIR²-compliant mental health survey dataset using `mlcroissant`. All data and metadata were referenced by their unique Croissant `@id`. We loaded and previewed survey records, filtered for high PHQ-9 scores, normalized the data, grouped by village, and visualized key statistics. This approach ensures reproducibility and clarity via strict schema and ID-based referencing. 

For further work, you may analyze additional record sets, link to documentation fields, or integrate the data in machine learning workflows.